# H100: temporal resolution × output/loss

Frozen MERT-v1-95M; 1 or 8 disjoint temporal means per original window. Compact 64-channel, three-Conv1D GELU residual head; scalar sigmoid/BCE or two-logit softmax/CE. Fixed C08 hyperparameters and seeds 41–43. Fresh coarse and fine controls share one extraction. Selection uses tune only; historical evaluation is reused and does not establish leaderboard performance.

In [ ]:
from temporal_resolution import main
main()

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
out=Path('results')
selection=json.loads((out/'selection.json').read_text())
reports=json.loads((out/'test_report.json').read_text())['results']
lookup={r['name']:r for r in reports}
fig,axes=plt.subplots(1,3,figsize=(14,4),layout='constrained')
summary=[]
for group in selection['summaries']:
    rows=[lookup[k] for k in group['candidates']]
    summary.append(dict(name=group['name'],tune_eer=group['music_eer_mean'],
        test_music_eer=float(np.mean([r['music_eer'] for r in rows])),
        test_file_eer=float(np.mean([r['metrics']['metrics']['file_eer'] for r in rows]))))
for ax,key,title in zip(axes,['tune_eer','test_music_eer','test_file_eer'],['Tune music EER','Historical music EER','Historical file EER']):
    bars=ax.bar([r['name'].replace('_','\n') for r in summary],[r[key] for r in summary],color=['#a6b3c2','#6c8eb4','#56a99a','#ee9b49'])
    ax.bar_label(bars,fmt='%.3f');ax.set_ylim(0,.65);ax.set_title(title+' (lower better)')
    ax.grid(axis='y',alpha=.2)
fig.savefig(out/'comparison.png',dpi=170)
plt.show()
(out/'summary.json').write_text(json.dumps(dict(groups=summary,winner=selection['winner']['name']),indent=2))
print(json.dumps(summary,indent=2))